# Direct uniformly magnetised cubes: cdfmm versus MagTense

This validation compares the direct all-to-all field of 10,000 identical, axis-aligned, uniformly magnetised cube sources at the same 10,000 centre points. Every target receives contributions from every source, including its finite cube self-field, for a total of 100 million source-target pairs.

The public MagTense Python API evaluates a finite prism at **points**. Accordingly, this comparison starts with the `UNIFORM_CUBOID -> POINT` backend. A single `CDFMM_BACKEND` setting selects portable CPU, oneMKL CPU, or CUDA execution. It measures both accuracy and persistent-geometry performance, but does not yet validate cdfmm's receiving-volume-averaged cuboid tensor. The two dense tensors are built sequentially to avoid holding both multi-gigabyte plans in memory at once.

In [ ]:
from importlib.metadata import version

import gc
import time

import cdfmm
import matplotlib.pyplot as plt
import numpy as np

try:
    from magtense import magstatics
except ImportError as exc:
    raise ImportError(
        "MagTense is optional. Create and activate the environment in "
        "environment-magtense.yml, then restart this kernel."
    ) from exc

print(f"MagTense version: {version('magtense')}")

## Shared physical problem

The 10,000 source centres form a centred $20\times20\times25$ lattice. A three-cell-width centre spacing prevents overlap. The target points are exactly these same centres, so the comparison includes each finite cube self-field as well as every cross-cube interaction. Deterministic non-collinear magnetisations exercise diagonal and off-diagonal tensor components. Both libraries use the same SI coordinates and magnetisations. cdfmm expects each source's **total magnetic moment**, so we pass $m=V M$.

In [ ]:
# ==================================================================
# Physical problem setup
# ==================================================================

# Change only this setting to select the cdfmm executor.
# Valid values: "normal-cpu", "mkl-cpu", and "cuda".
CDFMM_BACKEND = "cuda"
VALID_CDFMM_BACKENDS = {"normal-cpu", "mkl-cpu", "cuda"}
if CDFMM_BACKEND not in VALID_CDFMM_BACKENDS:
    raise ValueError(
        f"CDFMM_BACKEND must be one of {sorted(VALID_CDFMM_BACKENDS)}"
    )

MU0 = 4.0 * np.pi * 1.0e-7
MU0_MS_T = 1.2
MS = MU0_MS_T / MU0

CUBE_SIDE_M = 10.0e-9
CUBE_VOLUME_M3 = CUBE_SIDE_M**3
CENTRE_SPACING_M = 3.0 * CUBE_SIDE_M
N_CUBES = 1000
N_EVALUATION_POINTS = 1000
TIMING_REPEATS = 20
RELATIVE_L2_TOLERANCE = 5.0e-5
LATTICE_SHAPE = (10, 10, 10)
assert np.prod(LATTICE_SHAPE) == N_CUBES
assert N_EVALUATION_POINTS == N_CUBES

grid_indices = np.indices(LATTICE_SHAPE, dtype=np.float64)
lattice_coordinates = np.column_stack(
    [axis.reshape(-1) for axis in grid_indices]
)
lattice_coordinates -= 0.5 * (np.asarray(LATTICE_SHAPE) - 1.0)
cube_centres = np.ascontiguousarray(CENTRE_SPACING_M * lattice_coordinates)
n_cubes = len(cube_centres)

rng = np.random.default_rng(seed=20260820)
directions = rng.normal(size=(n_cubes, 3))
directions /= np.linalg.norm(directions, axis=1)[:, None]
magnetisations = np.ascontiguousarray(MS * directions)
total_moments = np.ascontiguousarray(
    CUBE_VOLUME_M3 * magnetisations
)

# Evaluate at every source centre. Uniform-cuboid self-interactions are
# finite and are deliberately retained by both implementations.
evaluation_points = cube_centres.copy()

print(f"Number of cubes:       {n_cubes}")
print(f"Cube side:             {CUBE_SIDE_M * 1e9:.1f} nm")
print(f"Centre spacing:        {CENTRE_SPACING_M * 1e9:.1f} nm")
print(f"Evaluation points:     {N_EVALUATION_POINTS:,}")
print(f"Source-target pairs:   {N_CUBES * N_EVALUATION_POINTS:,}")
print(f"Timing repeats:        {TIMING_REPEATS}")
print(f"cdfmm backend:         {CDFMM_BACKEND}")
print(f"Magnetisation norm:    {MS:.6e} A/m")

In [ ]:
# ==================================================================
# cdfmm dense direct all-to-all evaluation
# ==================================================================

cube_size = cdfmm.CuboidSize(
    CUBE_SIDE_M,
    CUBE_SIDE_M,
    CUBE_SIDE_M,
)

if CDFMM_BACKEND == "mkl-cpu" and not cdfmm.dense_direct_mkl_available():
    raise RuntimeError(
        "mkl-cpu requires a build configured with CDFMM_ENABLE_MKL=ON"
    )
if CDFMM_BACKEND == "cuda" and not cdfmm.cuda_dense_direct_available():
    raise RuntimeError(
        "cuda requires CDFMM_ENABLE_CUDA=ON and an available CUDA device"
    )

plan_arguments = dict(
    source_positions=cube_centres,
    target_positions=evaluation_points,
    source_geometry=cdfmm.SourceGeometry.UNIFORM_CUBOID,
    target_geometry=cdfmm.TargetGeometry.POINT,
    source_sizes=[cube_size],
)
cpu_backend = {
    "normal-cpu": cdfmm.DenseDirectBackend.PORTABLE,
    "mkl-cpu": cdfmm.DenseDirectBackend.ONE_MKL,
}.get(CDFMM_BACKEND)

start = time.perf_counter()
if CDFMM_BACKEND == "cuda":
    cdfmm_plan = cdfmm.CudaDenseDirectPlan(**plan_arguments)
    cdfmm_memory_location = "device"
else:
    cdfmm_plan = cdfmm.DenseDirectPlan(**plan_arguments)
    cdfmm_memory_location = "host"
cdfmm_setup_s = time.perf_counter() - start

def evaluate_cdfmm():
    if CDFMM_BACKEND == "cuda":
        return np.asarray(cdfmm_plan.evaluate(total_moments))
    return np.asarray(
        cdfmm_plan.evaluate(total_moments, backend=cpu_backend)
    )

# Warm the reusable matrix application before collecting timings.
evaluate_cdfmm()
cdfmm_evaluation_samples_s = []
for _ in range(TIMING_REPEATS):
    start = time.perf_counter()
    H_cdfmm = evaluate_cdfmm()
    cdfmm_evaluation_samples_s.append(time.perf_counter() - start)
cdfmm_evaluation_s = float(np.median(cdfmm_evaluation_samples_s))
cdfmm_tensor_memory_bytes = cdfmm_plan.tensor_memory_bytes
cdfmm_persistent_memory_bytes = (
    cdfmm_plan.persistent_device_bytes
    if CDFMM_BACKEND == "cuda"
    else cdfmm_tensor_memory_bytes
)

# Keep only the 10,000-vector result before constructing MagTense's larger
# nine-component tensor. This prevents both dense plans coexisting.
del cdfmm_plan
gc.collect()

In [ ]:
# ==================================================================
# MagTense prism all-to-all evaluation
# ==================================================================

start = time.perf_counter()
tiles = magstatics.Tiles(
    n=n_cubes,
    tile_type=2,  # Rectangular prism
    size=[CUBE_SIDE_M, CUBE_SIDE_M, CUBE_SIDE_M],
    offset=np.asfortranarray(cube_centres),
    rot=[0.0, 0.0, 0.0],
    M_rem=0.0,
)
tiles.M = np.asfortranarray(magnetisations)
magtense_points = np.asfortranarray(evaluation_points)
magtense_tensor = magstatics.get_demag_tensor(
    tiles,
    magtense_points,
)
magtense_setup_s = time.perf_counter() - start

# Reuse MagTense's cached tensor to time only changing-moment evaluation.
magstatics.get_H_field(tiles, magtense_points, magtense_tensor)
magtense_evaluation_samples_s = []
for _ in range(TIMING_REPEATS):
    start = time.perf_counter()
    H_magtense = np.asarray(
        magstatics.get_H_field(tiles, magtense_points, magtense_tensor)
    )
    magtense_evaluation_samples_s.append(time.perf_counter() - start)
magtense_evaluation_s = float(np.median(magtense_evaluation_samples_s))
magtense_tensor_memory_bytes = magtense_tensor.nbytes

# Accuracy only needs the resulting field after the timing loop.
del magtense_tensor
gc.collect()

## Field convention

cdfmm defines the physical field as $H=-\nabla\phi$ and its cube self-field is $H=-M/3$. The packaged MagTense 2.2.0 CPU implementation returns the same physical field convention at all 10,000 coincident source/target centres, so no sign conversion is applied.

In [ ]:
# ==================================================================
# Accuracy and performance summary
# ==================================================================

def relative_l2_error(actual, reference):
    return np.linalg.norm(actual - reference) / np.linalg.norm(reference)

difference = H_cdfmm - H_magtense
absolute_error = np.linalg.norm(difference, axis=1)
reference_magnitude = np.linalg.norm(H_magtense, axis=1)
pointwise_relative_error = absolute_error / np.maximum(
    reference_magnitude,
    np.finfo(float).eps,
)

relative_l2 = relative_l2_error(H_cdfmm, H_magtense)
pair_count = n_cubes * N_EVALUATION_POINTS
cdfmm_pairs_per_s = pair_count / cdfmm_evaluation_s
magtense_pairs_per_s = pair_count / magtense_evaluation_s

print("Accuracy")
print("--------")
print(f"Relative L2 error:                     {relative_l2:.6e}")
print(f"Maximum absolute vector error:        {absolute_error.max():.6e} A/m")
print(f"Maximum pointwise relative error:     {pointwise_relative_error.max():.6e}")
print()
print("Performance")
print("-----------")
print(f"Interactions per evaluation:          {pair_count:,}")
print(f"cdfmm backend:                        {CDFMM_BACKEND}")
print(f"cdfmm setup:                          {cdfmm_setup_s:.6f} s")
print(f"MagTense setup:                       {magtense_setup_s:.6f} s")
print(f"cdfmm median evaluation:              {1e3 * cdfmm_evaluation_s:.6f} ms")
print(f"MagTense median evaluation:           {1e3 * magtense_evaluation_s:.6f} ms")
print(f"cdfmm interaction throughput:         {cdfmm_pairs_per_s:.6e} pairs/s")
print(f"MagTense interaction throughput:      {magtense_pairs_per_s:.6e} pairs/s")
print(f"cdfmm cached tensor memory:           {cdfmm_tensor_memory_bytes / 1e9:.3f} GB")
print(f"cdfmm persistent {cdfmm_memory_location} memory:        {cdfmm_persistent_memory_bytes / 1e9:.3f} GB")
print(f"MagTense cached tensor memory:        {magtense_tensor_memory_bytes / 1e9:.3f} GB")

assert relative_l2 < RELATIVE_L2_TOLERANCE, (
    f"cdfmm and MagTense differ by {relative_l2:.3e}; "
    f"tolerance is {RELATIVE_L2_TOLERANCE:.3e}"
)

print()
print("PASS: the direct finite-cube fields agree within tolerance.")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
implementations = [f"cdfmm\n{CDFMM_BACKEND}", "MagTense"]
axes[0].bar(implementations, [cdfmm_setup_s, magtense_setup_s])
axes[0].set_ylabel("Setup time [s]")
axes[0].set_title("Geometry/tensor setup")
axes[0].grid(axis="y", alpha=0.3)
axes[1].bar(
    implementations,
    [1e3 * cdfmm_evaluation_s, 1e3 * magtense_evaluation_s],
)
axes[1].set_ylabel("Median evaluation time [ms]")
axes[1].set_title(f"Persistent evaluation ({TIMING_REPEATS} repeats)")
axes[1].grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

## Signed component error

The first error view is the direct componentwise difference $H_{\mathrm{cdfmm}}-H_{\mathrm{MagTense}}$. Positive and negative deviations remain visible rather than being collapsed into a norm.

In [ ]:
target_indices = np.arange(N_EVALUATION_POINTS)
component_labels = [r"$\Delta H_x$", r"$\Delta H_y$", r"$\Delta H_z$"]

fig, ax = plt.subplots(figsize=(12, 4))
for component, label in enumerate(component_labels):
    ax.plot(
        target_indices,
        difference[:, component],
        ".",
        markersize=1.5,
        alpha=0.65,
        rasterized=True,
        label=label,
    )
ax.axhline(0.0, color="black", linewidth=0.8)
ax.set_xlabel("Evaluation-point index")
ax.set_ylabel(r"$H_{\mathrm{cdfmm}}-H_{\mathrm{MagTense}}$ [A/m]")
ax.set_title("Signed field-component error")
ax.legend(ncol=3)
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

## Absolute vector error

The absolute error is $\lVert H_{\mathrm{cdfmm}}-H_{\mathrm{MagTense}}\rVert_2$ at each evaluation point.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.semilogy(
    target_indices,
    np.maximum(absolute_error, np.finfo(float).tiny),
    ".",
    markersize=1.5,
    alpha=0.65,
    rasterized=True,
)
ax.set_xlabel("Evaluation-point index")
ax.set_ylabel("Absolute vector error ||ΔH||₂ [A/m]")
ax.set_title("Absolute field error")
ax.grid(alpha=0.25, which="both")
fig.tight_layout()
plt.show()

## Relative vector error

The pointwise relative error divides the absolute vector error by $\lVert H_{\mathrm{MagTense}}\rVert_2$.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.semilogy(
    target_indices,
    np.maximum(pointwise_relative_error, np.finfo(float).tiny),
    ".",
    markersize=1.5,
    alpha=0.65,
    rasterized=True,
)
ax.set_xlabel("Evaluation-point index")
ax.set_ylabel("Relative vector error ||ΔH||₂ / ||H_MagTense||₂")
ax.set_title("Pointwise relative field error")
ax.grid(alpha=0.25, which="both")
fig.tight_layout()
plt.show()

## What this validates

This validates cdfmm's axis-aligned uniform-cuboid source tensor, the selected portable CPU, oneMKL CPU, or CUDA dense executor, all-to-all accumulation across 100 million pairs, total-moment normalisation, 10,000 finite cube self-terms, persistent tensor reuse, and Python binding against MagTense's analytical rectangular-prism calculation.

It does **not** yet test a cuboid FMM traversal, rotated cuboids, unequal side lengths, or the exact receiving-volume-averaged target. Only the backend selected by `CDFMM_BACKEND` is exercised in one notebook run. The natural next comparison is `TargetGeometry.VOLUME_AVERAGED_CUBOID` once an equivalent MagTense volume-average entry point is available.